In [1]:
"""
The LLM player offers a share in the division to the second player
LLM is used via API
"""

import os
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import RateLimitError 

from pathlib import Path
import re
import time
import pandas as pd

api_key = "" 
url = "https://foundation-models.api.cloud.ru/v1"

MAX_SEQ_LENGTH = 500
RUN_DIR = Path.cwd()
RES_DIR = str(RUN_DIR)+"/results"
DATA_DIR = "c:/Users/PC1/Documents/AgentsMD/Ультиматум/ЛЛМ играет (Ult)/"
TEMPERATURE = 0.01 #0.7
#base_model = "openai/gpt-oss-120b"
base_model = "Qwen/Qwen3-235B-A22B-Instruct-2507"
#ROLE = "system"

# The choice of a user-based or system-user interaction model depends on the experimental design. The system-user model is the classic chat model, while the user model is the assistant-based interaction model.
ROLE = "user"

data = "Ult_8_eng.csv"
#data = '//AdvUlt_5_AnoChar_BisChar_eng_2.csv'
prompt_ds = pd.read_csv(str(DATA_DIR) + data, delimiter=';')

client = OpenAI(
    api_key=api_key,
    base_url=url
)
@retry(
    stop=stop_after_attempt(5),  # Максимум 5 попыток
    wait=wait_exponential(multiplier=1, min=5, max=10),  # Экспоненциальная задержка (1-10 секунд)
    retry=retry_if_exception_type(RateLimitError) # Повторять только при RateLimitError
)
def generate_response(prompt):
    """Генерирует ответ, автоматически повторяя при RateLimitError."""
    try:
      response = client.chat.completions.create(
          model=base_model,
          messages=[{"role": ROLE, "content": prompt}],  
          # messages=[{"role": "system", "content": prompt_role}]
          # messages=[{"role": "user", "content": prompt_content}]
          max_tokens=MAX_SEQ_LENGTH,
          temperature=TEMPERATURE,
          presence_penalty=0,
      )
      return response.choices[0].message.content#.strip()
    except Exception as e:
        print(f"Error during llm request: {e}")
        raise # re-raise чтобы tenacity правильно обработал

ds_prompt = prompt_ds["Промпт"]
#ds_proffession = ds_proffession.dropna()
ds_prompt = ds_prompt.dropna()   

table = pd.DataFrame()
table["prompts"] = prompt_ds["Промпт"]
table = table.dropna()
print("len(table) ", len(table))

num_of_example = 0
start_index = 0
start = time.time()
numofiterations = 10

ans_names = ["answers_"+str(i) for i in range(0,numofiterations)]
ans_num_names = ["answers_num_"+str(i) for i in range(0,numofiterations)]

for i in range(0,numofiterations): # количество повторов вопроса
  cnt = 0
  answers = []
  answers_num = []

  for prompt_t in ds_prompt:
    response = "-1"
    try:
        response = generate_response(prompt_t)
        if response:
            print(f"{cnt} Ответ: {response}")
    except Exception as e:
        print(f"Не удалось получить ответ после нескольких попыток: {e}")
    cnt += 1

    answer = response
    answers.append(answer)
    if answer is not None:
    
        pattern = r"Here's my answer: +([0-1](?:\.\d+)?|\.\d+)"
        #pattern = r"Here's my advice: +([0-1](?:\.\d+)?|\.\d+)"
        print("---------")
        
        match = re.search(pattern, answer)  
        if match:
          try:
            answers_num.append(float(match.group(1))) # рассматриваем только первое совпадение
            print(float(match.group(1)))
          except ValueError: print("no share in answer")
        else: 
          answers_num.append(-1)
          print("no match")
    else:
       print("LLM вернула невалидный ответ (None).")  
       answers_num.append(-1)
    print("---------------------------------------------------------------------------")
  print("len(answers) ", len(answers))
  print("len(answers_num) ", len(answers_num))
  table[ans_names[i]] = answers
  table[ans_num_names[i]] = answers_num

header = time.strftime('%Y%m%d-%H%M')
table.to_excel(str(RES_DIR)+ "/" +base_model[:4]+header+'_ans_Ult8_user_t01.xlsx', engine="xlsxwriter", index=False)

ev_time = (time.time()-start)/60
print(f"evaluated in {ev_time} min.") 

def save_list_to_txt(my_list, filename):
  """
  """
  try:
    with open(filename, 'w', encoding='utf-8') as file:
      for item in my_list:
        file.write(str(item) + '\n')  
  except Exception as e:
    print(f" {e}")

save_list_to_txt([
("MODEL_MAX_SEQ_LEN ", MAX_SEQ_LENGTH), 
("model_name", base_model),
("data", data),
("TEMPERATURE", TEMPERATURE),
("time", ev_time),
("role", ROLE),
], str(RES_DIR) + "/" +base_model[:4]+header+"_tech_info.txt")


len(table)  121
0 Ответ: ###Here's my answer: 0.6

As a manager, I must consider fairness, motivation, and long-term collaboration. Offering 60% to the other manager demonstrates respect for their role, encourages acceptance, and reflects strong interpersonal and leadership skills. It aligns with the principle of fostering cooperation and mutual respect, essential for organizational success. A generous split reduces the risk of rejection, ensuring both parties benefit. This decision reflects strategic thinking, effective delegation, and emotional intelligence—core managerial competencies.
---------
0.6
---------------------------------------------------------------------------
1 Ответ: ###Here's my answer: 0.6

Professionals bring specialized expertise, innovation, and independent problem-solving that significantly enhance outcomes. Recognizing their value, a larger share incentivizes collaboration and acknowledges their contribution. Offering 60% builds trust, aligns with their high a